# SO-app escritorio en Kaggle

Requisito: haber corrido el cuaderno principal (rootfs + Chrome).
Levanta Xvnc + openbox + Chrome visible y publica el escritorio por tunel.
Abre el link `.../vnc.html` en el telefono. Toque = clic.


In [ ]:
import os
import re
import shutil
import socket
import subprocess
import tarfile
import time
import urllib.request

DIR = "/kaggle/working/vmdata/proot"
ROOTFS = DIR + "/rootfs"
VNC_PORT, NOVNC_PORT = 5900, 6080
os.environ["PATH"] = ("/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:"
                      + os.environ.get("PATH", ""))
os.environ["LC_ALL"] = "C"
os.environ["LANG"] = "C"

MARK = os.path.join(ROOTFS, ".soapp-ok")


def log(m=""):
    print(m, flush=True)


def run(cmd, check=False):
    log("$ " + " ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError("fallo rc=%d" % p.returncode)
    return p.returncode


def grab(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       text=True, errors="replace")
    return r.returncode, r.stdout


def proot(*args):
    return ["/usr/bin/proot", "-R", ROOTFS, "-b", "/dev", "-b", "/proc",
            "-b", "/sys", "-w", "/"] + list(args)


def port_open(port):
    try:
        socket.create_connection(("127.0.0.1", port), timeout=2).close()
        return True
    except OSError:
        return False


if os.path.isfile(MARK):
    log("rootfs OK")
elif os.path.isfile(os.path.join(ROOTFS, "bin", "sh")):
    open(MARK, "w").write("ok\n")
    log("rootfs previo (sin marcador), adoptado")
else:
    raise SystemExit("ERROR: no hay rootfs (recycle lo borro?) - corre primero la celda principal")

log("=== [1/4] paquetes graficos ===")
run(proot("/bin/sh", "-c", "apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true update -qq"))
run(proot("/bin/sh", "-c", "apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -y tigervnc-standalone-server openbox xterm || true"))
for b in ["Xvnc", "openbox-session"]:
    rc, _ = grab(proot("/bin/sh", "-c", "command -v " + b))
    if rc != 0:
        raise SystemExit("ERROR: no quedo instalado " + b)
log("Xvnc + openbox OK")

log("=== [2/4] noVNC + websockify ===")
NOVNC = os.path.join(DIR, "novnc")
if not os.path.isfile(os.path.join(NOVNC, "vnc.html")):
    log("descargando noVNC...")
    urllib.request.urlretrieve("https://github.com/novnc/noVNC/archive/refs/heads/master.tar.gz", "/tmp/novnc.tgz")
    shutil.rmtree(NOVNC, ignore_errors=True)
    os.makedirs(DIR, exist_ok=True)
    with tarfile.open("/tmp/novnc.tgz") as t:
        t.extractall("/tmp/novnc-x")
    import glob as _g
    shutil.move(_g.glob("/tmp/novnc-x/*/")[0], NOVNC)
    log("noVNC OK")
else:
    log("noVNC existente, reuso")
if shutil.which("websockify") is None:
    log("instalando websockify...")
    run(["pip", "install", "-q", "websockify"], check=True)

log("=== [3/4] escritorio ===")
GXE = {"DISPLAY": ":0", "HOME": "/root", "LC_ALL": "C",
       "PATH": "/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"}
run(["/usr/bin/pkill", "-f", "Xvnc :0"])
run(["/usr/bin/pkill", "-f", "websockify.*6080"])
time.sleep(2)
xv = open("/tmp/xvnc.log", "w")
subprocess.Popen(proot("/bin/sh", "-c",
    "rm -f /tmp/.X0-lock; Xvnc :0 -geometry 1280x800 -depth 24 -SecurityTypes None"),
    stdout=xv, stderr=subprocess.STDOUT, start_new_session=True,
    env={**os.environ, **GXE})
ok = False
for _ in range(30):
    if port_open(VNC_PORT):
        ok = True
        break
    time.sleep(2)
if not ok:
    run(["/usr/bin/tail", "-15", "/tmp/xvnc.log"])
    raise SystemExit("ERROR: Xvnc no levanto")
log("Xvnc UP")
subprocess.Popen(proot("/bin/sh", "-c", "DISPLAY=:0 openbox-session"),
                 stdout=open("/tmp/openbox.log", "w"), stderr=subprocess.STDOUT,
                 start_new_session=True, env={**os.environ, **GXE})
time.sleep(3)
subprocess.Popen(proot("/bin/sh", "-c",
    "DISPLAY=:0 LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu "
    "/opt/google/chrome/chrome --no-sandbox --disable-dev-shm-usage --disable-gpu "
    "--no-first-run --window-size=1280,800 about:blank"),
    stdout=open("/tmp/chrome-gui.log", "w"), stderr=subprocess.STDOUT,
    start_new_session=True, env={**os.environ, **GXE})
log("openbox + Chrome lanzados")
ws = open("/tmp/websockify.log", "w")
subprocess.Popen(["websockify", "--web", NOVNC, str(NOVNC_PORT), "127.0.0.1:%d" % VNC_PORT],
                 stdout=ws, stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(3)
log("noVNC UP" if port_open(NOVNC_PORT) else "ERROR: websockify no levanto")

log("=== [4/4] tunel al escritorio ===")
CFD = "/tmp/cloudflared"
if not os.access(CFD, os.X_OK):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CFD)
    os.chmod(CFD, 0o755)
run(["/usr/bin/pkill", "-f", "cloudflared tunnel --url http://127.0.0.1:%d" % NOVNC_PORT])
time.sleep(2)
tlog = open("/tmp/tunnel-vnc.log", "w")
subprocess.Popen([CFD, "tunnel", "--url", "http://127.0.0.1:%d" % NOVNC_PORT],
                 stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
URL = ""
for _ in range(45):
    with open("/tmp/tunnel-vnc.log", errors="replace") as f:
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
    if m:
        URL = m.group(0)
        break
    time.sleep(2)
if not URL:
    run(["/usr/bin/tail", "-15", "/tmp/tunnel-vnc.log"])
    raise SystemExit("ERROR: sin URL de tunel")
log("ESCRITORIO: " + URL + "/vnc.html")
log("Abre ese link en el telefono: veras el escritorio Ubuntu con Chrome. Toque = clic.")
